In [1]:
import os, torch
import numpy as np
from PIL import Image
from glob import glob
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from skimage.morphology import label, skeletonize, erosion, remove_small_objects

from fil_finder import FilFinder2D
import astropy.units as u

prec_msk_root_BraTS = '/data/data2/motoko_WSSS_data/Brats20/msk/'
mask_filelist = glob(f'{prec_msk_root_BraTS}/*.png')


def get_longest_skeleton(raw_msk):
    labelled, max_id = label(np.where(raw_msk!=255, raw_msk,0),return_num=True)
    skeleton = np.zeros_like(raw_msk)
    for segment_id in range(1,max_id+1):
        segment = erosion(labelled==segment_id)
        if segment.max()  == 0:
            continue
        skeleton += skeletonize(segment).astype('uint8')
    
    try:
        fil = FilFinder2D(skeleton, distance= u.pc, mask=skeleton)
        fil.preprocess_image(flatten_percent = 50)
        fil.create_mask(border_masking=True, verbose=False, use_existing_mask=True)
        fil.medskel(verbose=False)
        fil.analyze_skeletons(branch_thresh=10* u.pix, skel_thresh=10 * u.pix, prune_criteria='length')
        return fil.skeleton_longpath
        
    except:
        return None


def get_skeleton_cls(skeleton_msk, raw_msk):
    labelled_skeleton, max_id = label(skeleton_msk,return_num=True)
    skeleton_with_cls = np.zeros_like(labelled_skeleton)
    
    for i in range(1, max_id+1):
        selected_segment = labelled_skeleton == i
        skeleton_px = raw_msk[selected_segment]
        assert skeleton_px.min() == skeleton_px.max()
        skeleton_with_cls += skeleton_px.min()*selected_segment.astype('uint8')
    
    return skeleton_with_cls


if not os.path.exists('/data/data2/motoko_WSSS_data/Brats20/scribRaw/'):
    os.mkdir('/data/data2/motoko_WSSS_data/Brats20/scribRaw/')
    # for sub_dir in os.listdir(prec_msk_root_BraTS):
    #     os.mkdir(f'/data/data2/motoko_WSSS_data/BraTS/scrib_sam_v2/{sub_dir}')
    
    

In [2]:
from random import sample
from tqdm.notebook import tqdm
error_list = []

for msk_pth in tqdm(mask_filelist):
    
    raw_msk = np.array(Image.open(msk_pth).convert('L')).clip(max=1)
    longest_skeleton = get_longest_skeleton(raw_msk)
    
    if longest_skeleton is None:
        print(msk_pth)
        error_list.append(msk_pth)
        continue
    
    skeleton_with_cls = get_skeleton_cls(longest_skeleton, raw_msk).astype('uint8')
    imageio.imwrite(msk_pth.replace('/msk/','/scribRaw/'), skeleton_with_cls)

  0%|          | 0/16896 [00:00<?, ?it/s]

/home/yz696/.local/lib/python3.11/site-packages/fil_finder/filfinder2D.py:142: UserWarning: No beam width given. Using 0 pixels.
  warnings.warn("No beam width given. Using 0 pixels.")
/home/yz696/.local/lib/python3.11/site-packages/astropy/units/quantity.py:666: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/yz696/.local/lib/python3.11/site-packages/astropy/units/quantity.py:666: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/yz696/.local/lib/python3.11/site-packages/fil_finder/filfinder2D.py:296: UserWarning: Using inputted mask. Skipping creation of anew mask.
  warnings.warn("Using inputted mask. Skipping creation of a"
/home/yz696/.local/lib/python3.11/site-packages/fil_finder/filament.py:326: UserWarning: Graph pruning reached max iterations.
  warnings.warn("Graph pruning reached max iterations.")


/data/data2/motoko_WSSS_data/Brats20/msk/Train089_s35.png
/data/data2/motoko_WSSS_data/Brats20/msk/Train291_s63.png
/data/data2/motoko_WSSS_data/Brats20/msk/Train128_s64.png
/data/data2/motoko_WSSS_data/Brats20/msk/Train320_s78.png
/data/data2/motoko_WSSS_data/Brats20/msk/Train188_s119.png
/data/data2/motoko_WSSS_data/Brats20/msk/Train161_s115.png
/data/data2/motoko_WSSS_data/Brats20/msk/Train188_s120.png
